In [5]:
import cupy as cp
print(cp.cuda.runtime.runtimeGetVersion())

13020


In [1]:
import subprocess
result = subprocess.run(
    ['/home/nehadesigar/my_env/bin/python', '-c', 'import cupy; print(cupy.cuda.runtime.runtimeGetVersion())'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

12090




In [2]:
# THIS CODE USES CUPY; FOR MORE THAN 500 GRIDPOINTS
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from PIL import Image
from numba import jit, prange

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import os
from scipy.signal import find_peaks
from scipy.integrate import quad
from scipy.optimize import root_scalar
from matplotlib.animation import PillowWriter, FuncAnimation
import ipywidgets as widgets
from IPython.display import display

print("Fin")

Fin


In [ ]:
# Independent parameters (free to edit)

Na = 0.5 # Units: M 
T = 303.15 # Units: K
valence = 4
duration = 250 * 10**5 # In timesteps of dt
gridpoints = 1024 # Number of points
dx = 10 # Units: nm
dt = 1.E-5 # Units: sec
rho_mean = 9E-5 # Initial mean density of nanostar A, found by spinodal (rho dense + rho dilute)/2 for the value of T used
save_interval = 10**5

grid_length = dx * gridpoints # Total length (nm)
inv_dx2 = 1.0 / (dx * dx)

# Establishes constants
K = 1.0E6 # Units: nm^5 
M = 1 # Units: (nm s)^-1
B2 = 2190 # Units: nm^3
vb = 1.66 # Units: nm^3
kB = 1.314E-23 * 0.24 # Units: cal/K (1J=0.24cal)
mol = 6.02E23
dHa = -42000 # Units: cal/mol 
dS1 = 1.84 * cp.log(Na) # Units: cal/mol K
dS0 = -120 # Units: cal/mol K at 1M NaCl
floor = 1E-12 # Minimum value for arrays
num_saves = duration // save_interval + 1 # Number of saved values

Da = vb * cp.exp(-(dHa - T * (dS0 + dS1)) / (mol * kB * T))
Db = Da

print("Fin")

In [ ]:
# Initializes array of density values
cp.random.seed(7) # Opens a random number generator instance, seed 7

rho_A = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints))) # Creates rho values around the mean with slight randomness
rho_A = cp.maximum(rho_A, 1.E-10) # Prevents negative densities
rho_B = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints))) 
rho_B = cp.maximum(rho_B, 1.E-10) 
rho_AB = rho_mean * (1.0 + 0.01 * cp.random.uniform(low=-1, high=1, size=(gridpoints, gridpoints))) 
rho_AB = cp.maximum(rho_AB, 1.E-10) 

initial_mass = cp.sum(rho_A) + cp.sum(rho_B) + cp.sum(rho_AB)
n = rho_A.shape[0]


def laplacian_2d(function_array):
    """
    Computes the 2D Laplacian of a function, given an array representing that function
    """
    return ( #Uses the inbuilt roll which does allow for periodic boundary conditions
        cp.roll(function_array,  1, axis=1) +
        cp.roll(function_array, -1, axis=1) +
        cp.roll(function_array,  1, axis=0) +
        cp.roll(function_array, -1, axis=0) -
        4.0 * function_array) * inv_dx2




beta_mu_kernel_A = cp.ElementwiseKernel(
    'float64 rho_A, float64 rho_B, float64 rho_AB, float64 lap_rho_A',
    'float64 output',
    f'''
    double rho = rho_A + rho_B + rho_AB;
    double Ca = (rho_A + 0.5 * rho_AB) * {valence} * {Da};
    double Xa = (-1.0 + sqrt(1.0 + 4.0 * Ca)) / (2.0 * Ca);
    output = 2.0 * {B2} * rho
           + log(rho_A)
           + {valence} * log(Xa)
           - {K} * lap_rho_A;
    ''',
    'beta_mu_kernel_A'
)

beta_mu_kernel_B = cp.ElementwiseKernel(
    'float64 rho_A, float64 rho_B, float64 rho_AB, float64 lap_rho_B',
    'float64 output',
    f'''
    double rho = rho_A + rho_B + rho_AB;
    double Cb = (rho_B + 0.5 * rho_AB) * {valence} * {Db};
    double Xb = (-1.0 + sqrt(1.0 + 4.0 * Cb)) / (2.0 * Cb * {Db});
    output = 2.0 * {B2} * rho
           + log(rho_B)
           + {valence} * log(Xb)
           - {K} * lap_rho_B;
    ''',
    'beta_mu_kernel_B'
)

beta_mu_kernel_AB = cp.ElementwiseKernel(
    'float64 rho_A, float64 rho_B, float64 rho_AB, float64 lap_rho_AB',
    'float64 output',
    f'''
    double rho = rho_A + rho_B + rho_AB;
    double Ca = (rho_A + 0.5 * rho_AB) * {valence} * {Da};
    double Cb = (rho_B + 0.5 * rho_AB) * {valence} * {Db};
    double Xa = (-1.0 + sqrt(1.0 + 4.0 * Ca)) / (2.0 * Ca);
    double Xb = (-1.0 + sqrt(1.0 + 4.0 * Cb)) / (2.0 * Ca * {Db});
    output = 2.0 * {B2} * rho
           + log(rho_AB)
           + {valence} / 2.0 * (log(Xa) + log(Xb))
           - {K} * lap_rho_AB;
    ''',
    'beta_mu_kernel_AB'
)




def compute_step_three(rho_A, rho_B, rho_AB):
    lap_A  = laplacian_2d(rho_A)
    lap_B  = laplacian_2d(rho_B)
    lap_AB = laplacian_2d(rho_AB)

    mu_A  = beta_mu_kernel_A(rho_A, rho_B, rho_AB, lap_A)
    mu_B  = beta_mu_kernel_B(rho_A, rho_B, rho_AB, lap_B)
    mu_AB = beta_mu_kernel_AB(rho_A, rho_B, rho_AB, lap_AB)

    rho_A_step  = dt * M * laplacian_2d(mu_A)
    rho_B_step  = dt * M * laplacian_2d(mu_B)
    rho_AB_step = dt * M * laplacian_2d(mu_AB)

    return rho_A_step, rho_B_step, rho_AB_step


# Initializes arrays for saving rho
num_saves = duration // save_interval + 1

rho_A_total_array = cp.zeros((num_saves, gridpoints, gridpoints)) # Third dimension added for 2D grid
rho_B_total_array = cp.zeros((num_saves, gridpoints, gridpoints))
rho_AB_total_array = cp.zeros((num_saves, gridpoints, gridpoints))

rho_A_total_array[0] = rho_A
rho_B_total_array[0] = rho_B
rho_AB_total_array[0] = rho_AB

save_index = 1

# Tracks the mass over time to ensure conservation
mass_history = []
time_history = []


for step in range(duration):

    # Iterates to find new value of rho
    rho_A_step, rho_B_step, rho_AB_step = compute_step_three(rho_A, rho_B, rho_AB)
    rho_A += rho_A_step
    rho_B += rho_B_step
    rho_AB += rho_AB_step

    # Adds the new density to the array of densities + checks mass conservation every 10^6 steps
    if step % (save_interval) == 0:
        rho_A_total_array[save_index] = rho_A
        rho_B_total_array[save_index] = rho_B
        rho_AB_total_array[save_index] = rho_AB
        save_index += 1

        total_mass = cp.sum(rho_A) + cp.sum(rho_B) + cp.sum(rho_AB)

        mass_history.append(total_mass)
        time_history.append(step * dt)

        rho_A = cp.maximum(rho_A, floor)        
        rho_B = cp.maximum(rho_B, floor)
        rho_AB = cp.maximum(rho_AB, floor)


        print(f"Progress: {(step/(save_interval))} out of {duration/(save_interval)}")

In [ ]:
rho_A_np = rho_A.get()
rho_B_np = rho_B.get()
rho_AB_np = rho_AB.get()

normalized_A = (rho_A_np - np.min(rho_A_np)) / (np.max(rho_A_np) - np.min(rho_A_np))
normalized_B = (rho_B_np - np.min(rho_B_np)) / (np.max(rho_B_np) - np.min(rho_B_np))
normalized_AB = (rho_AB_np - np.min(rho_AB_np)) / (np.max(rho_AB_np) - np.min(rho_AB_np))

rgb_image = np.dstack((normalized_A, normalized_B, normalized_AB))
t_final = (save_index - 1) * save_interval * dt

plt.figure(figsize=(8, 8))
plt.imshow(rgb_image, extent=[0, grid_length, 0, grid_length], origin='lower')
plt.title(f"State at time t = {t_final:.0f} s")
plt.xlabel("x (nm)")
plt.ylabel("y (nm)")
plt.xticks([0, grid_length/4, grid_length/2, (3*grid_length)/4, grid_length])
plt.yticks([0, grid_length/4, grid_length/2, (3*grid_length)/4, grid_length])

plt.tight_layout()
plt.savefig("rho_final.png", dpi=150)
plt.show()

In [ ]:
def find_perimeter(rho):

    # Creates binary array: 0 for pixel in dilute phase, 1 for pixel in dense phase
    threshold = (rho.max() + rho.min()) / 2
    binary_mask = (rho >= threshold).astype(int)

    # Pads with periodic boundaries
    padded = np.pad(binary_mask, pad_width=1, mode='wrap')

    # Checks to see if each pixel has at least one dilute neighbor
    neighbor_sum = (padded[:-2, 1:-1] + padded[2:,  1:-1] + padded[1:-1, :-2] + padded[1:-1, 2:])

    # Finds dense pixels with dilute neighbors + counts
    interface_mask = (binary_mask == 1) & (neighbor_sum < 4)
    interface_count = int(interface_mask.sum())

    return binary_mask, interface_mask, interface_count


def calc_free_energy(rho_A, rho_B, rho_AB, Xa, Xb, v0 = 1):
    '''
    Finds free energy of a multi-component system using F = integral_V [ f({rho_i}) + K/2 sum_i (gradient rho)^2 ] dV

    Using f_ref = rho_tot * log(v_0 * rho_tot) - rho_tot + B2 * rho_tot^2 + sum_i [rho_i * log(rho_i/rho_tot)]

    Note that the final term IS included in this!

    Using f_b = rho_A * valence * (log(Xa) + (1-Xa)/2) + rho_B * valence * (log(Xb) + (1-Xb)/2) 
              + rho_AB * valence/2 * (log(Xa) + (1-Xa)/2) + rho_AB * valence/2 * (log(Xb) + (1-Xb)/2)
    
    For AAAA, BBBB, and AABB
    '''

    # Calculates the total density
    rho_tot = rho_A + rho_B + rho_AB
    
    # To prevent error
    rho_A_f = np.maximum(rho_A, floor)
    rho_B_f = np.maximum(rho_B, floor)
    rho_AB_f = np.maximum(rho_AB, floor)
    rho_tot_f = np.maximum(rho_tot, floor)

    # Finds f_ref including the final term
    f_ref = (rho_tot_f * np.log(v0 * rho_tot_f) - rho_tot_f + B2 * rho_tot_f**2 +
            rho_A_f * np.log(rho_A_f / rho_tot_f) +  rho_B_f * np.log(rho_B_f / rho_tot_f) + rho_AB_f * np.log(rho_AB_f / rho_tot_f))

    # Finds f_b for each component and adds
    f_b = (rho_A_f * 4 * (np.log(Xa) + (1-Xa)/2) +
            rho_B_f * 4 * (np.log(Xb) + (1-Xb)/2) +
            rho_AB_f * 2 * (np.log(Xa) + (1-Xa)/2) + rho_AB_f * 2 * (np.log(Xb) + (1-Xb)/2))
    

    # Finds the local free energy at each point (f{rho_i})
    f_local = f_ref + f_b
    
    # Calculates the gradient term K/2 * (gradient rho)^2
    grad_ax, grad_ay = np.gradient(rho_A_f)
    grad_bx, grad_by = np.gradient(rho_B_f)
    grad_abx, grad_aby = np.gradient(rho_B_f)
    
    f_gradient = 0.5 * K * (grad_ax**2 + grad_ay**2 + grad_bx**2 + grad_by**2 + grad_abx**2 + grad_aby**2)
    
    # Total free energy density
    f_total = f_local + f_gradient
    
    # Integrate over volume in 2D (multiply by dx and sum)
    F_total = np.trapezoid(np.trapezoid(f_total, dx=dx, axis= 1), axis = 0)
    
    return F_total

In [ ]:
# Note that this calculates the free energy cost comparing the coexisting system to a system that is all three all dense phases and all three all dilute

# To prevent error
rho_A_f = np.maximum(rho_A, floor)
rho_B_f = np.maximum(rho_B, floor)
rho_AB_f = np.maximum(rho_AB, floor)

#Finds interfacial area for each rho
binary_mask_A, interface_mask_A, interface_count_A = find_perimeter(rho_A_f)
interfacial_area_A = interface_count_A * dx

binary_mask_B, interface_mask_B, interface_count_B = find_perimeter(rho_B_f)
interfacial_area_B = interface_count_B * dx

binary_mask_AB, interface_mask_AB, interface_count_AB = find_perimeter(rho_AB_f)
interfacial_area_AB = interface_count_AB * dx

# Extracts dilute/dense phase densities and creates homogenous systems
rho_A_dilute = np.min(rho_A_f) 
rho_A_dense = np.max(rho_A_f)
rho_B_dilute = np.min(rho_B_f)
rho_B_dense = np.max(rho_B_f)
rho_AB_dilute = np.min(rho_AB_f)
rho_AB_dense = np.max(rho_AB_f)

rho_A_dilute_system = np.full((gridpoints, gridpoints), rho_A_dilute)
rho_A_dense_system = np.full((gridpoints, gridpoints), rho_A_dense)
rho_B_dilute_system = np.full((gridpoints, gridpoints), rho_B_dilute)
rho_B_dense_system = np.full((gridpoints, gridpoints), rho_B_dense)
rho_AB_dilute_system = np.full((gridpoints, gridpoints), rho_AB_dilute)
rho_AB_dense_system = np.full((gridpoints, gridpoints), rho_AB_dense)

# Finds Xa for coexisting system, dense, and dilute
Xa_final_coex = np.maximum(Xa, floor)
Xb_final_coex = np.maximum(Xb, floor)

CaDa_dense = 4*rho_A_dense_system*Da + 2*rho_AB_dense_system*Da
CaDa_dense_floored = np.maximum(CaDa_dense, floor)
Xa_final_dense = (-1 + np.sqrt(1 + 4 * CaDa_dense_floored)) / (2 * CaDa_dense_floored)
CaDb_dense = 4*rho_B_dense_system*Db + 2*rho_AB_dense_system*Db
CaDb_dense_floored = np.maximum(CaDb_dense, floor)
Xb_final_dense = (-1 + np.sqrt(1 + 4 * CaDb_dense_floored)) / (2 * CaDb_dense_floored)

CaDa_dilute = 4*rho_A_dilute_system*Da + 2*rho_AB_dilute_system*Da
CaDa_dilute_floored = np.maximum(CaDa_dilute, floor)
Xa_final_dilute = (-1 + np.sqrt(1 + 4 * CaDa_dilute_floored)) / (2 * CaDa_dilute_floored)
CaDb_dilute = 4*rho_B_dilute_system*Db + 2*rho_AB_dilute_system*Db
CaDb_dilute_floored = np.maximum(CaDb_dilute, floor)
Xb_final_dilute = (-1 + np.sqrt(1 + 4 * CaDb_dilute_floored)) / (2 * CaDb_dilute_floored)

# Finds free energy of dilute, dense, and coexisting systems
F_dilute = calc_free_energy(rho_A_dilute_system, rho_B_dilute_system, rho_AB_dilute_system, Xa_final_dilute, Xb_final_dilute)
F_dense = calc_free_energy(rho_A_dense_system, rho_B_dense_system, rho_AB_dense_system, Xa_final_dense, Xb_final_dense)
F_coex = calc_free_energy(rho_A_f, rho_B_f, rho_AB_ff, Xa_final_coex, Xb_final_coex)

surface_tension = (F_coex - F_dilute - F_dense) / (interfacial_area_A + interfacial_area_B + interfacial_area_AB)
print(f"The surface tension is {surface_tension:.6e} for T = {T} in a 2D system with 3 components (AAAA, BBBB, and AABB with Cx = {Cx}) initialized with a uniform distribution")

In [ ]:
# This version compares each of AAAA, BBBB, AABB to systems in which they are dilute and dense (isolated)

def calc_free_energy_isolated (rho, Xa, v0 = 1):
    '''
    Finds free energy of a system using F = integral_V [ f({rho_i}) + K/2 sum_i (gradient rho)^2 ] dV

    Using f_ref = rho * log(v_0 * rho) - rho + B2 * rho^2 + {sum_i [rho_i log(rho_i/rho_tot)]}

    The last term will always be 0 for a 1-component system (log 1 = 0), so it not included here

    Using f_b = rho_A * valence * (log(Xa) + (1-Xa)/2)
    
    Note that this will vary by mixture-- eq shown is for AAAA only
    '''

    # Floors rho to prevent error
    rho_floored = np.maximum(rho, floor)

    # Finds f_ref and f_b
    f_ref = rho_floored * np.log(v0 * rho_floored) - rho_floored + B2 * rho_floored**2
    f_b = rho_floored * 4 * (np.log(Xa) + (1-Xa)/2)

    # Finds the local free energy at each point (f{rho_i})
    f_local = f_ref + f_b
    
    # Calculates the gradient term K/2 * (gradient rho)^2
    grad_x, grad_y = np.gradient(rho)
    f_gradient = 0.5 * K * (grad_x**2 + grad_y**2)
    
    # Total free energy density
    f_total = f_local + f_gradient
    
    # Integrate over volume in 1D (multiply by dx and sum)
    F_total = np.trapezoid(np.trapezoid(f_total, dx=dx, axis= 1), axis = 0)
    
    return F_total


F_dilute_A = calc_free_energy_isolated (rho_A_dilute_system, Xa_final_dilute)
F_dense_A = calc_free_energy_isolated (rho_A_dense_system, Xa_final_dense)
F_A = calc_free_energy_isolated (rho_A_f, Xa_final_coex)
surface_tension_A = (F_A - F_dilute_A - F_dense_A) / (interfacial_area_A)
print(f"The surface tension is {surface_tension_A:.6e} for AAAA in a multicomponent system")

F_dilute_B = calc_free_energy_isolated (rho_B_dilute_system, Xb_final_dilute)
F_dense_B = calc_free_energy_isolated (rho_B_dense_system, Xb_final_dense)
F_B = calc_free_energy_isolated (rho_B_f, Xb_final_coex)
surface_tension_B = (F_B - F_dilute_B - F_dense_B) / (interfacial_area_B)
print(f"The surface tension is {surface_tension_B:.6e} for BBBB in a multicomponent system")

F_dilute_AB = calc_free_energy_isolated (rho_AB_dilute_system, 0.5 * Xa_final_dilute + 0.5 *Xb_final_dilute)
F_dense_AB = calc_free_energy_isolated (rho_AB_dense_system,  0.5 * Xa_final_dense + 0.5 *Xb_final_dense)
F_AB = calc_free_energy_isolated (rho_AB_f,  0.5 * Xa_final_coex + 0.5 *Xb_final_coex)
surface_tension_AB = (F_AB - F_dilute_AB - F_dense_AB) / (interfacial_area_AB)
print(f"The surface tension is {surface_tension_AB:.6e} for AABB in a multicomponent system")